In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
from json import load as load_json
import re
from bs4.element import Tag

In [ ]:
_Delay = 2.5

with open("config.json", "r") as f:
        headers = load_json(f)
headers

In [ ]:
with open("../../data/raw/urls/teams_url2.txt", "r") as f:
    teams_url = f.read().splitlines()
teams_url[:5]

columns = [
        "ID",
        "Name",
        "Franchise",
        "From",
        "To",
        "Years (Yrs)",
        # "Conference",
        # "Division",
]

dc_url = "https://www.basketball-reference.com/leagues/NBA_2024.html"

dc_columns = [
	"Name",
        "Conference",
        "Division",
]

teams_data_p1 = pd.DataFrame(columns = columns)
teams_data_p2 = pd.DataFrame(columns = dc_columns)

In [ ]:
def get_data(row: Tag, data_stat: str):
        try:
                return row.find(attrs={"data-stat": data_stat}).a.text.strip()
        except:
                try:
                        return row.find(attrs={"data-stat": data_stat}).text.strip()
                except:
                        pass

def team_info(url: str, headers: dict):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = {}
        
        while (True):
                try:
                        info_tag = soup.select("#meta > div")[1]
                        body_tag = soup.select_one(f"#{url.split('/')[4]} > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        
        # `Name`
        result[columns[1]] = get_data(body_tag.select_one("tr"), "team_name")
        
        # Franchise
        try:
                result[columns[2]] = info_tag.h1.select_one("span").text
        except:
                pass
        
        # From
        # To
        # Years (Yrs)
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Seasons' in text))
                if strong_tag != None:
                        pt = strong_tag.find_parent('p')
                        
                        txt = pt.get_text(separator=' ', strip=True)
                        count_match = re.search(r'(\d+)\s*;', txt)
                        if (count_match is None):
                                count_match = re.search(r'(\d+)\s+NBA', txt)
                        result[columns[5]] = int(count_match.group(1)) if count_match else None
                        seasons = re.findall(r'(\d{4})-(\d{2})', txt)
                        result[columns[3]] = int(seasons[0][0]) + 1
                        result[columns[4]] = int(seasons[1][0]) + 1
        except:
                pass
        
        time.sleep(_Delay)
        return result

In [ ]:
teams_data = pd.DataFrame(columns = columns)
id = 0

In [ ]:
for i in tqdm(range(len(teams_url))):
        result = team_info(teams_url[i], headers)
        if result == None:
                id += 1
                continue
        
        result[columns[0]] = id
        teams_data_p1.loc[id] = result
        id += 1
        
teams_data_p1

In [ ]:
def get_info(body: Tag) -> pd.DataFrame:
        info = body.select("tr")
        div = ""
        result = pd.DataFrame(columns = dc_columns)
        for row in info:
                row_info = {}
                if ("thead" in row.get("class", default = "")):
                        div = row.th.strong.text
                        continue
                else:
                        row_info[dc_columns[0]] = get_data(row = row, data_stat = "team_name")
                        row_info[dc_columns[2]] = div
                result.loc[len(result)] = row_info
        return result

def get_div_and_conf(url: str, headers: dict):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = pd.DataFrame()
        
        while True:
                try:
                        Eastern_Division_body = soup.select_one("#divs_standings_E > tbody")
                        Western_Division_body = soup.select_one("#divs_standings_W > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
                
        
        ec_df = get_info(Eastern_Division_body)
        ec_df[dc_columns[1]] = "Eastern Conference"
        wc_df = get_info(Western_Division_body)
        wc_df[dc_columns[1]] = "Western Conference"
        result = pd.concat([ec_df, wc_df], ignore_index = True)
        
        time.sleep(_Delay)
        return result

teams_data_p2 = get_div_and_conf(dc_url, headers=headers)
teams_data_p2

In [ ]:
result = teams_data_p1.merge(teams_data_p2, on = columns[1])
result.to_csv("../../data/raw/teams_data.csv", index = False)
result